# 1. Imports

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [3]:
import glob
import re

In [4]:
# !tree

In [5]:
def split_mun(df, col="Municipios", split_char=" "):
    partes = df[col].str.split(split_char, n=1)
    df = df.assign(cod_mun=partes.str[0], nombre=partes.str[1])
    return df[["cod_mun", "nombre", *df.columns.drop(["cod_mun", "nombre", col])]]

# 2. Datos de padrón

In [6]:
base_path = "datos/padron/"

In [7]:
PROV_CLM = ("02 Albacete", "13 Ciudad Real", "16 Cuenca", "19 Guadalajara", "45 Toledo")
PRE_CLM = ("02", "13", "16", "19", "45")

In [8]:
provincias_clm = [
    "02 Albacete",
    "13 Ciudad Real",
    "16 Cuenca",
    "19 Guadalajara",
    "45 Toledo"
]

## 2.1. Cifras de población

> **DATOS DEL 1 DE ENERO DE CADA AÑO**

In [9]:
path = "poblacion_municipios/cifra_oficial_pob_"
dfs = []

for suffix in ["ab.csv", "cr.csv", "cu.csv", "gu.csv", "to.csv"]:
    dfs.append(pd.read_csv(base_path+path+suffix, sep=";", decimal=",", thousands="."))

pob = pd.concat(dfs, ignore_index=True).rename(columns={"Periodo": "periodo"})
pob = split_mun(pob)
pob.head()

,cod_mun,nombre,Sexo,periodo,Total
0,02001,Abengibre,Total,2025,753.0
1,02001,Abengibre,Total,2024,759.0
2,02001,Abengibre,Total,2023,760.0
3,02001,Abengibre,Total,2022,739.0
4,02001,Abengibre,Total,2021,748.0


In [10]:
# pob["Sexo"].unique()

In [11]:
# pob.isna().sum()

In [12]:
# pob.loc[pob.isna().any(axis=1), "cod_mun"].drop_duplicates()

In [13]:
# pob.loc[pob.isna().any(axis=1), ["cod_mun", "periodo"]].drop_duplicates()

In [14]:
# pob.loc[pob.isna().any(axis=1), ["periodo"]].value_counts()

In [15]:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.pivot.html#pandas.DataFrame.pivot
pob = pob.pivot(index=["cod_mun", "nombre", "periodo"], columns="Sexo", values="Total").reset_index("nombre")
pob.head()

Sexo               nombre   Hombres   Mujeres     Total
cod_mun periodo                                        
02      1996     Albacete  178130.0  180880.0  359010.0
        1997     Albacete       NaN       NaN       NaN
        1998     Albacete  178058.0  180539.0  358597.0
        1999     Albacete  179210.0  181811.0  361021.0
        2000     Albacete  180311.0  182952.0  363263.0

> NOTA: 02 Albacete se refiere a la provincia, 02003 a la capital de provincia

In [16]:
df = pob.copy().rename(columns={"Hombres": "hombres", "Mujeres": "mujeres", "Total": "total", "En Extranjero": "en_ext"})
df.head()

Sexo               nombre   hombres   mujeres     total
cod_mun periodo                                        
02      1996     Albacete  178130.0  180880.0  359010.0
        1997     Albacete       NaN       NaN       NaN
        1998     Albacete  178058.0  180539.0  358597.0
        1999     Albacete  179210.0  181811.0  361021.0
        2000     Albacete  180311.0  182952.0  363263.0

## 2.2. Estadística del Padrón continuo

> **DATOS DEL 1 DE ENERO DE CADA AÑO**

### 2.2.1. Relación lugar de nacimiento y lugar de residencia

> DATOS DEL 1 DE ENERO DE CADA AÑO

#### 2023-2022

In [17]:
path    = base_path+"estadistica_padron/2003-2022/pob_nacimiento_residencia_*.csv"
files   = glob.glob(path)
rel     = pd.concat([pd.read_csv(f, sep=";", decimal=",", thousands=".") for f in files], ignore_index=True)
rel["Municipios"] = rel["Municipios"].fillna(rel["Provincias"])
rel["Periodo"] = rel["Periodo"].str.split(" ").str[-1]
rel = rel.rename(columns={"Relación entre lugar de nacimiento y residencia": "Nacimiento"})
rel = rel.sort_values(["Municipios", "Periodo"]).reset_index(drop=True)
rel = split_mun(rel).rename(columns={"Periodo": "periodo"})
rel.head()

,cod_mun,nombre,Sexo,Provincias,Nacimiento,periodo,Total
0,02,Albacete,Total,02 Albacete,Total,2003,376556
1,02,Albacete,Total,02 Albacete,En España,2003,360779
2,02,Albacete,Total,02 Albacete,En la misma Comunidad Autónoma,2003,326784
3,02,Albacete,Total,02 Albacete,Misma Comunidad Autónoma. Misma Provincia.,2003,317436
4,02,Albacete,Total,02 Albacete,Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio,2003,243607


In [18]:
# rel.loc[rel.isna().any(axis=1), "cod_mun"].drop_duplicates()

In [19]:
rel["Nacimiento"].unique()

<ArrowStringArray>
[                                                        'Total',
                                                     'En España',
                                'En la misma Comunidad Autónoma',
                    'Misma Comunidad Autónoma. Misma Provincia.',
    'Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio',
 'Misma Comunidad Autónoma. Misma Provincia. Distinto Municipio',
                  'Misma Comunidad Autónoma. Distinta Provincia',
                                'En distinta Comunidad Autónoma',
                                              'En el extranjero']
Length: 9, dtype: str

In [20]:
rename = {
    "En España":                                                        "n_esp",
    "En el extranjero":                                                 "n_ext",
    "En la misma Comunidad Autónoma":                                   "n_en_ca",
    "En distinta Comunidad Autónoma":                                   "n_dif_ca",
    "Misma Comunidad Autónoma. Misma Provincia.":                       "n_en_prov",
    "Misma Comunidad Autónoma. Distinta Provincia":                     "n_dif_prov",                  
    "Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio":       "n_en_mun",
    "Misma Comunidad Autónoma. Misma Provincia. Distinto Municipio":    "n_dif_mun",
}

In [21]:
rel["periodo"] = rel["periodo"].astype("int64") # hay que cuadrar dtypes o el join no se hace bien
rel = (
    rel.loc[
        (rel["Sexo"] == "Total") & (rel["Nacimiento"] != "Total"), 
        ["cod_mun", "Nacimiento", "periodo", "Total"]
    ]
    .pivot(index=["cod_mun", "periodo"], columns="Nacimiento", values="Total")
    .rename(columns=rename)
    .reindex(columns=rename.values())
)
rel = rel[["n_ext", "n_dif_ca", "n_dif_prov", "n_dif_mun", "n_en_mun"]]
rel.head()

Nacimiento       n_ext  n_dif_ca  n_dif_prov  n_dif_mun  n_en_mun
cod_mun periodo                                                  
02      2003     15777     33995        9348      73829    243607
        2004     18509     34283        9451      73507    243698
        2005     22906     34816        9543      73198    244177
        2006     25073     35251        9675      72932    244727
        2007     28594     35833        9788      72551    245344

In [22]:
print(df.index.dtypes)
print(rel.index.dtypes)

cod_mun    object
periodo     int64
dtype: object
cod_mun    object
periodo     int64
dtype: object


In [23]:
df = df.join(rel, how="left")
df[df["n_ext"].notna()].head()

nombre   hombres   mujeres     total    n_ext  n_dif_ca  \
cod_mun periodo                                                              
02      2003     Albacete  188139.0  188417.0  376556.0  15777.0   33995.0   
        2004     Albacete  189743.0  189705.0  379448.0  18509.0   34283.0   
        2005     Albacete  192624.0  192016.0  384640.0  22906.0   34816.0   
        2006     Albacete  194348.0  193310.0  387658.0  25073.0   35251.0   
        2007     Albacete  196772.0  195338.0  392110.0  28594.0   35833.0   

                 n_dif_prov  n_dif_mun  n_en_mun  
cod_mun periodo                                   
02      2003         9348.0    73829.0  243607.0  
        2004         9451.0    73507.0  243698.0  
        2005         9543.0    73198.0  244177.0  
        2006         9675.0    72932.0  244727.0  
        2007         9788.0    72551.0  245344.0

#### 2000-2002

In [24]:
prov_dict = {
    "ab": "02",
    "cr": "13",
    "cu": "16",
    "gu": "19",
    "to": "45",
}

In [25]:
path    = base_path+"estadistica_padron/[0-9][0-9][0-9][0-9]/pob_nacimiento_residencia_*.csv"
files   = glob.glob(path)
temp = [pd.read_csv(f, sep=";", decimal=",", thousands=".", na_values="..").assign(
    Periodo=int(re.search(r"estadistica_padron/(\d{4})/pob_nacimiento_residencia_.*\.csv$", f).group(1)),
    cod_prov=(prov_dict[re.search(r"estadistica_padron/\d{4}/pob_nacimiento_residencia_(.*)\.csv$", f).group(1)])
) for f in files]
rel     = pd.concat(temp, ignore_index=True)
rel["Total"] = rel["Total"].fillna(0)
rel = rel.rename(columns={"Relación lugar de nacimiento y de residencia": "Nacimiento"})
rel = rel.sort_values(["Municipios", "Periodo"]).reset_index(drop=True)
rel = split_mun(rel).rename(columns={"Periodo": "periodo"})
rel["cod_mun"] = np.where(rel["cod_mun"] == "Total", rel["cod_prov"], rel["cod_mun"])
rel = rel.sort_values(["cod_mun", "periodo"]).reset_index(drop=True)
rel.head()

,cod_mun,nombre,Sexo,Nacimiento,Total,periodo,cod_prov
0,02,NaN,Ambos sexos,Total,363263.0,2000,02
1,02,NaN,Ambos sexos,Nacidos en España,359034.0,2000,02
2,02,NaN,Ambos sexos,En la misma Comunidad Autónoma,325900.0,2000,02
3,02,NaN,Ambos sexos,Misma Comunidad Autónoma. Misma Provincia,316883.0,2000,02
4,02,NaN,Ambos sexos,Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio,244339.0,2000,02


In [26]:
# rel[rel["Total"] == ".."]

In [27]:
rel["Nacimiento"].unique()

<ArrowStringArray>
[                                                       'Total',
                                            'Nacidos en España',
                               'En la misma Comunidad Autónoma',
                    'Misma Comunidad Autónoma. Misma Provincia',
   'Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio',
 'Misma Comunidad Autónoma.Misma Provincia. Distinto Municipio',
                 'Misma Comunidad Autónoma. Distinta Provincia',
                               'En distinta Comunidad Autónoma',
                                     'Nacidos en el Extranjero']
Length: 9, dtype: str

In [28]:
rename = {
    "Nacidos en España":                                                "n_esp",
    "Nacidos en el Extranjero":                                         "n_ext",
    "En la misma Comunidad Autónoma":                                   "n_en_ca",
    "En distinta Comunidad Autónoma":                                   "n_dif_ca",
    "Misma Comunidad Autónoma. Misma Provincia":                        "n_en_prov",
    "Misma Comunidad Autónoma. Distinta Provincia":                     "n_dif_prov",                  
    "Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio":       "n_en_mun",
    "Misma Comunidad Autónoma.Misma Provincia. Distinto Municipio":     "n_dif_mun",
}

In [29]:
rel.head()

,cod_mun,nombre,Sexo,Nacimiento,Total,periodo,cod_prov
0,02,NaN,Ambos sexos,Total,363263.0,2000,02
1,02,NaN,Ambos sexos,Nacidos en España,359034.0,2000,02
2,02,NaN,Ambos sexos,En la misma Comunidad Autónoma,325900.0,2000,02
3,02,NaN,Ambos sexos,Misma Comunidad Autónoma. Misma Provincia,316883.0,2000,02
4,02,NaN,Ambos sexos,Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio,244339.0,2000,02


In [30]:
rel = rel.loc[
        (rel["Sexo"] == "Ambos sexos") & (rel["Nacimiento"] != "Total") & (rel["cod_mun"] != "Total"), 
        ["cod_mun", "Nacimiento", "periodo", "Total"]
].reset_index(drop=True)
rel.head()

,cod_mun,Nacimiento,periodo,Total
0,02,Nacidos en España,2000,359034.0
1,02,En la misma Comunidad Autónoma,2000,325900.0
2,02,Misma Comunidad Autónoma. Misma Provincia,2000,316883.0
3,02,Misma Comunidad Autónoma. Misma Provincia. Mismo Municipio,2000,244339.0
4,02,Misma Comunidad Autónoma.Misma Provincia. Distinto Municipio,2000,72544.0


In [31]:
rel["periodo"].value_counts()

periodo
2001    7344
2002    7344
2000    7312
Name: count, dtype: int64

In [32]:
# rel[["Municipios", "Periodo", "Nacimiento"]].value_counts()

In [33]:
rel = (
    rel.pivot(index=["cod_mun", "periodo"], columns="Nacimiento", values="Total")
    .rename(columns=rename)
    .reindex(columns=rename.values())
)
rel[list(rename.values())] = rel[list(rename.values())].astype("float")
rel.head()

Nacimiento          n_esp    n_ext   n_en_ca  n_dif_ca  n_en_prov  n_dif_prov  \
cod_mun periodo                                                                 
02      2000     359034.0   4229.0  325900.0   33134.0   316883.0      9017.0   
        2001     359723.0   7560.0  326387.0   33336.0   317274.0      9113.0   
        2002     359819.0  11968.0  326258.0   33561.0   317098.0      9160.0   
02001   2000        969.0      7.0     922.0      47.0      908.0        14.0   
        2001        980.0      7.0     931.0      49.0      916.0        15.0   

Nacimiento       n_en_mun  n_dif_mun  
cod_mun periodo                       
02      2000     244339.0    72544.0  
        2001     242631.0    74643.0  
        2002     242906.0    74192.0  
02001   2000        767.0      141.0  
        2001        765.0      151.0

In [34]:
df.update(rel)
df[(df["n_ext"].notna())].head()

nombre   hombres   mujeres     total    n_ext  n_dif_ca  \
cod_mun periodo                                                              
02      2000     Albacete  180311.0  182952.0  363263.0   4229.0   33134.0   
        2001     Albacete  182826.0  184457.0  367283.0   7560.0   33336.0   
        2002     Albacete  185458.0  186329.0  371787.0  11968.0   33561.0   
        2003     Albacete  188139.0  188417.0  376556.0  15777.0   33995.0   
        2004     Albacete  189743.0  189705.0  379448.0  18509.0   34283.0   

                 n_dif_prov  n_dif_mun  n_en_mun  
cod_mun periodo                                   
02      2000         9017.0    72544.0  244339.0  
        2001         9113.0    74643.0  242631.0  
        2002         9160.0    74192.0  242906.0  
        2003         9348.0    73829.0  243607.0  
        2004         9451.0    73507.0  243698.0

### 2.2.2. Nacionalidad y edad

#### 2003-2022

In [35]:
path    = base_path+"estadistica_padron/2003-2022/pob_nacionalidad_edad_*.csv"
files   = glob.glob(path)
ed      = pd.concat([pd.read_csv(f, sep=";", decimal=",", thousands=".") for f in files], ignore_index=True)
ed["Municipios"]    = ed["Municipios"].fillna(ed["Provincias"])
ed["Periodo"]       = ed["Periodo"].str.split(" ").str[-1]
ed = ed.rename(columns={"Edad (grandes grupos)": "Edad"})
ed = ed.sort_values(["Municipios", "Periodo"]).reset_index(drop=True)
ed = split_mun(ed).rename(columns={"Periodo": "periodo"})
ed.head()

,cod_mun,nombre,Sexo,Provincias,Nacionalidad,Edad,periodo,Total
0,02,Albacete,Total,02 Albacete,Total,Todas las edades,2003,376556
1,02,Albacete,Total,02 Albacete,Total,Menos de 16 años,2003,64971
2,02,Albacete,Total,02 Albacete,Total,De 16 a 64 años,2003,243724
3,02,Albacete,Total,02 Albacete,Total,65 y más años,2003,67861
4,02,Albacete,Total,02 Albacete,Española,Todas las edades,2003,363238


In [36]:
ed = ed.loc[
    (ed["Nacionalidad"] == "Total") & (ed["Sexo"] == "Total") & (ed["Edad"] != "Todas las edades"),
    ["cod_mun", "periodo", "Edad", "Total"]
]
ed.head()

,cod_mun,periodo,Edad,Total
1,02,2003,Menos de 16 años,64971
2,02,2003,De 16 a 64 años,243724
3,02,2003,65 y más años,67861
37,02,2004,Menos de 16 años,64491
38,02,2004,De 16 a 64 años,246642


In [37]:
# ed.loc[ed.isna().any(axis=1), "cod_mun"].drop_duplicates()

In [38]:
ed["periodo"] = ed["periodo"].astype("int64")
ed = ed.pivot(index=["cod_mun", "periodo"], columns="Edad", values="Total")
ed = ed.rename(columns={"Menos de 16 años": "ed_0_15", "De 16 a 64 años": "ed_16_64", "65 y más años": "ed_65+"})
ed.head()

Edad             ed_65+  ed_16_64  ed_0_15
cod_mun periodo                           
02      2003      67861    243724    64971
        2004      68315    246642    64491
        2005      67913    252139    64588
        2006      69503    253474    64681
        2007      69580    257699    64831

In [39]:
df = df.join(ed, how="left")
df[df["ed_0_15"].notna()].head()

nombre   hombres   mujeres     total    n_ext  n_dif_ca  \
cod_mun periodo                                                              
02      2003     Albacete  188139.0  188417.0  376556.0  15777.0   33995.0   
        2004     Albacete  189743.0  189705.0  379448.0  18509.0   34283.0   
        2005     Albacete  192624.0  192016.0  384640.0  22906.0   34816.0   
        2006     Albacete  194348.0  193310.0  387658.0  25073.0   35251.0   
        2007     Albacete  196772.0  195338.0  392110.0  28594.0   35833.0   

                 n_dif_prov  n_dif_mun  n_en_mun   ed_65+  ed_16_64  ed_0_15  
cod_mun periodo                                                               
02      2003         9348.0    73829.0  243607.0  67861.0  243724.0  64971.0  
        2004         9451.0    73507.0  243698.0  68315.0  246642.0  64491.0  
        2005         9543.0    73198.0  244177.0  67913.0  252139.0  64588.0  
        2006         9675.0    72932.0  244727.0  69503.0  253474.0  64681.0  
        2007         9788.0    72551.0  245344.0  69580.0  257699.0  64831.0

#### 2000-2002

In [40]:
path    = base_path+"estadistica_padron/[0-9][0-9][0-9][0-9]/pob*_edad_*.csv"
files   = glob.glob(path)
# print(files)
temp    = [pd.read_csv(f, sep=";", decimal=",", thousands=".", na_values="..").assign(
    Periodo=int(re.search(r"estadistica_padron/(\d{4})/pob.*edad_.*\.csv$", f).group(1)),
    cod_prov=(prov_dict[re.search(r"estadistica_padron/\d{4}/pob.*edad_(.*)\.csv$", f).group(1)]),    
) for f in files]
ed      = pd.concat(temp, ignore_index=True)
ed["Total"] = ed["Total"].fillna(0)
ed = ed.rename(columns={"Nacionalidad (español/extranjero)": "Nacionalidad"})
ed = ed.rename(columns={"Edad (grandes grupos)": "Edad"})
ed = ed.sort_values(["Municipios", "Periodo"]).reset_index(drop=True)
ed = split_mun(ed).rename(columns={"Periodo": "periodo"})
ed["cod_mun"] = np.where(ed["cod_mun"] == "Total", ed["cod_prov"], ed["cod_mun"])
ed = ed.sort_values(["cod_mun", "periodo"]).reset_index(drop=True)
ed.head()

,cod_mun,nombre,Edad,Sexo,Total,periodo,cod_prov,Nacionalidad
0,02,NaN,Total,Ambos sexos,363263.0,2000,02,NaN
1,02,NaN,Total,Varones,180311.0,2000,02,NaN
2,02,NaN,Total,Mujeres,182952.0,2000,02,NaN
3,02,NaN,Menores de 16 años,Ambos sexos,65952.0,2000,02,NaN
4,02,NaN,Menores de 16 años,Varones,33907.0,2000,02,NaN


In [41]:
ed = ed.loc[
    ((ed["Nacionalidad"] == "Total") | ed["Nacionalidad"].isna())
    & (ed["Sexo"] == "Ambos sexos") & (ed["Edad"] != "Total") & (ed["cod_mun"] != "Total"),
    ["cod_mun", "periodo", "Edad", "Total"]
]
ed.head()

,cod_mun,periodo,Edad,Total
3,02,2000,Menores de 16 años,65952.0
6,02,2000,De 16 a 64 años,232715.0
9,02,2000,De 65 y más años,64596.0
15,02,2001,Menores de 16 años,65080.0
18,02,2001,De 16 a 64 años,236246.0


In [42]:
ed["periodo"] = ed["periodo"].astype("int64")
ed = ed.pivot(index=["cod_mun", "periodo"], columns="Edad", values="Total")
ed = ed.rename(columns={"Menores de 16 años": "ed_0_15", "De 16 a 64 años": "ed_16_64", "De 65 y más años": "ed_65+"})
ed[["ed_0_15", "ed_16_64", "ed_65+"]] = ed[["ed_0_15", "ed_16_64", "ed_65+"]].astype("float") 
ed.head()

Edad             ed_16_64   ed_65+  ed_0_15
cod_mun periodo                            
02      2000     232715.0  64596.0  65952.0
        2001     236246.0  65957.0  65080.0
        2002     240043.0  67020.0  64724.0
02001   2000        633.0    219.0    124.0
        2001        632.0    231.0    124.0

In [43]:
df.update(ed)
df[df["ed_0_15"].notna()].head()

nombre   hombres   mujeres     total    n_ext  n_dif_ca  \
cod_mun periodo                                                              
02      2000     Albacete  180311.0  182952.0  363263.0   4229.0   33134.0   
        2001     Albacete  182826.0  184457.0  367283.0   7560.0   33336.0   
        2002     Albacete  185458.0  186329.0  371787.0  11968.0   33561.0   
        2003     Albacete  188139.0  188417.0  376556.0  15777.0   33995.0   
        2004     Albacete  189743.0  189705.0  379448.0  18509.0   34283.0   

                 n_dif_prov  n_dif_mun  n_en_mun   ed_65+  ed_16_64  ed_0_15  
cod_mun periodo                                                               
02      2000         9017.0    72544.0  244339.0  64596.0  232715.0  65952.0  
        2001         9113.0    74643.0  242631.0  65957.0  236246.0  65080.0  
        2002         9160.0    74192.0  242906.0  67020.0  240043.0  64724.0  
        2003         9348.0    73829.0  243607.0  67861.0  243724.0  64971.0  
        2004         9451.0    73507.0  243698.0  68315.0  246642.0  64491.0

## 2.4. Estadística de variaciones residenciales

> **DATOS DE CADA AÑO NATURAL**

In [44]:
files = glob.glob(base_path+"estadistica_variaciones_residenciales/*.csv")
temp = [pd.read_csv(f, sep=";", decimal=",", thousands=".").assign(Periodo=int(re.search(r"_(\d{4})\.csv$", f).group(1))) for f in files]
vars = pd.concat(temp, ignore_index=True)
del temp
vars.head()

,Municipio,Tipo de Saldo,Nacionalidad,Sexo,Total,Periodo
0,Total,Total Altas,Total,Total,2137195,2017
1,Total,Total Altas,Total,Hombres,1072185,2017
2,Total,Total Altas,Total,Mujeres,1065010,2017
3,Total,Total Altas,Españoles,Total,1260680,2017
4,Total,Total Altas,Españoles,Hombres,629777,2017


In [45]:
print(provincias_clm)

['02 Albacete', '13 Ciudad Real', '16 Cuenca', '19 Guadalajara', '45 Toledo']


In [46]:
tipos = [
    "Total Bajas", "Bajas Exteriores", "Bajas Interiores",
    "Total Altas", "Altas Exteriores", "Altas Interiores",
]

vars = vars.loc[
    (vars["Nacionalidad"] == "Total") & (vars["Sexo"] == "Total") & (vars["Tipo de Saldo"].isin(tipos))
    & (
        vars["Municipio"].str.startswith(("02", "13", "16", "19", "45"))
    )
, ["Municipio", "Periodo", "Tipo de Saldo", "Total"]].sort_values(["Municipio", "Total"]).reset_index(drop=True)
print(vars["Municipio"].str[:2].unique())

<ArrowStringArray>
['02', '13', '16', '19', '45']
Length: 5, dtype: str


In [47]:
vars.loc[vars.isna().any(axis=1), "Municipio"].drop_duplicates()

Series([], Name: Municipio, dtype: str)

In [48]:
vars[(vars["Municipio"].str.startswith("02003")) & (vars["Tipo de Saldo"] == "Altas Interiores") & (vars["Periodo"] > 2016)].sort_values("Periodo")
# vars[(vars["Municipio"].str.contains("-")) & (vars["Tipo de Saldo"] == "Altas Interiores") & (vars["Periodo"] > 2017)].sort_values("Periodo").sort

,Municipio,Periodo,Tipo de Saldo,Total
110,02003-Albacete,2017,Altas Interiores,3102
115,02003-Albacete,2018,Altas Interiores,3802
94,02003 Albacete,2019,Altas Interiores,3844
90,02003 Albacete,2020,Altas Interiores,2815
91,02003 Albacete,2021,Altas Interiores,3407


Antes de 2019 se usa "-" como separador del código y el nombre, a partir de ese año se usa " ", que coincide con el formato en el resto de datos. Hay que reemplazarlo.

In [49]:
vars = vars.rename(columns={"Municipio": "Municipios"})
vars["Municipios"] = np.where(vars["Periodo"] >= 2019, vars["Municipios"], vars["Municipios"].str.replace("-", " ", 1))
vars[(vars["Municipios"].str.startswith("02003")) & (vars["Tipo de Saldo"] == "Altas Interiores") & (vars["Periodo"] > 2016)].sort_values("Periodo")

,Municipios,Periodo,Tipo de Saldo,Total
110,02003 Albacete,2017,Altas Interiores,3102
115,02003 Albacete,2018,Altas Interiores,3802
94,02003 Albacete,2019,Altas Interiores,3844
90,02003 Albacete,2020,Altas Interiores,2815
91,02003 Albacete,2021,Altas Interiores,3407


In [50]:
rename = {
    "Total Altas": "altas_total", "Total Bajas": "bajas_total",
    "Bajas Exteriores": "bajas_ext", "Bajas Interiores": "bajas_int",
    "Altas Exteriores": "altas_ext", "Altas Interiores": "altas_int",

}
vars = split_mun(vars).rename(columns={"Periodo": "periodo"}).sort_values(by=["cod_mun", "periodo"], ascending=True)
vars = vars.pivot(index=["cod_mun", "periodo"], columns="Tipo de Saldo", values="Total").rename(columns=rename)
vars.head()

Tipo de Saldo    altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2015             4         22          2         18           26   
        2016             4         30         26         30           34   
        2017             5         30          1         13           35   
        2018             6         59          1         22           65   
        2019             3         19          0         43           22   

Tipo de Saldo    bajas_total  
cod_mun periodo               
02001   2015              20  
        2016              56  
        2017              14  
        2018              23  
        2019              43

In [51]:
df = df.join(vars, how="left")
df[df["altas_total"].notna()].head()

nombre  hombres  mujeres  total  n_ext  n_dif_ca  \
cod_mun periodo                                                        
02001   2015     Abengibre    386.0    396.0  782.0   64.0      42.0   
        2016     Abengibre    366.0    395.0  761.0   49.0      47.0   
        2017     Abengibre    359.0    389.0  748.0   50.0      49.0   
        2018     Abengibre    363.0    394.0  757.0   55.0      53.0   
        2019     Abengibre    379.0    411.0  790.0   57.0      56.0   

                 n_dif_prov  n_dif_mun  n_en_mun  ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                               
02001   2015           22.0      117.0     537.0   232.0     499.0     51.0   
        2016           24.0      116.0     525.0   233.0     478.0     50.0   
        2017           21.0      111.0     517.0   225.0     479.0     44.0   
        2018           21.0      115.0     513.0   224.0     485.0     48.0   
        2019           22.0      127.0     528.0   231.0     507.0     52.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2015           4.0       22.0        2.0       18.0         26.0   
        2016           4.0       30.0       26.0       30.0         34.0   
        2017           5.0       30.0        1.0       13.0         35.0   
        2018           6.0       59.0        1.0       22.0         65.0   
        2019           3.0       19.0        0.0       43.0         22.0   

                 bajas_total  
cod_mun periodo               
02001   2015            20.0  
        2016            56.0  
        2017            14.0  
        2018            23.0  
        2019            43.0

# 3. Censos 

> **DATOS DEL 1 DE ENERO DE CADA AÑO**

In [52]:
base_path = "datos/censos_demograficos/censo_anual_poblacion/"

## 3.1. Población

### 3.1.1. Relaciones nacimiento residencia

In [53]:
path = base_path+"poblacion_2021-2025/pob_edad_nacimiento_residencia.csv"
df.head()

nombre   hombres   mujeres     total   n_ext  n_dif_ca  \
cod_mun periodo                                                             
02      1996     Albacete  178130.0  180880.0  359010.0     NaN       NaN   
        1997     Albacete       NaN       NaN       NaN     NaN       NaN   
        1998     Albacete  178058.0  180539.0  358597.0     NaN       NaN   
        1999     Albacete  179210.0  181811.0  361021.0     NaN       NaN   
        2000     Albacete  180311.0  182952.0  363263.0  4229.0   33134.0   

                 n_dif_prov  n_dif_mun  n_en_mun   ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                                
02      1996            NaN        NaN       NaN      NaN       NaN      NaN   
        1997            NaN        NaN       NaN      NaN       NaN      NaN   
        1998            NaN        NaN       NaN      NaN       NaN      NaN   
        1999            NaN        NaN       NaN      NaN       NaN      NaN   
        2000         9017.0    72544.0  244339.0  64596.0  232715.0  65952.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02      1996           NaN        NaN        NaN        NaN          NaN   
        1997           NaN        NaN        NaN        NaN          NaN   
        1998           NaN        NaN        NaN        NaN          NaN   
        1999           NaN        NaN        NaN        NaN          NaN   
        2000           NaN        NaN        NaN        NaN          NaN   

                 bajas_total  
cod_mun periodo               
02      1996             NaN  
        1997             NaN  
        1998             NaN  
        1999             NaN  
        2000             NaN

In [54]:
pob = pd.read_csv(path, sep=";", decimal=",", thousands=".")
pob = pob.loc[
    pob["Municipios"].str.startswith(("02", "13", "16", "19", "45"))
].rename(columns={"Relacion entre lugar de nacimiento y residencia": "Relación"})
# print(pob["Municipios"].str[:2].unique())
pob.head()

,Total Nacional,Provincias,Municipios,Sexo,Edad,Relación,Periodo,Total
19440,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2025,747.0
19441,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2024,760.0
19442,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2023,768.0
19443,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2022,739.0
19444,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2021,752.0


In [55]:
pob.loc[pob.isna().any(axis=1), "Municipios"].drop_duplicates()

Series([], Name: Municipios, dtype: str)

In [56]:
print(pob["Relación"].unique())

<ArrowStringArray>
[                                                      'Total',
                            'Mismo municipio al de residencia',
                    'Distinto municipio de la misma provincia',
                    'Distinta provincia de la misma comunidad',
                                          'Distinta comunidad',
 'Nacido en el extranjero o en antiguos territorios españoles']
Length: 6, dtype: str


In [57]:
rename = {
    "Nacido en el extranjero o en antiguos territorios españoles":  "n_ext",
    "Distinta comunidad":                                           "n_dif_ca",
    "Distinta provincia de la misma comunidad":                     "n_dif_prov",
    "Distinto municipio de la misma provincia":                     "n_dif_mun",                  
    "Mismo municipio al de residencia":                             "n_en_mun",
}

pob = split_mun(pob).rename(columns={"Periodo": "periodo"})
pob.head()
pob = pob.loc[
    (pob["Sexo"] == "Total") & (pob["Edad"] == "Todas las edades") & (pob["Relación"] != "Total")
    , ["cod_mun", "periodo", "Relación", "Total"]
].pivot(index=["cod_mun", "periodo"], columns="Relación", values="Total").rename(columns=rename)
pob.head()

Relación         n_dif_ca  n_dif_prov  n_dif_mun  n_en_mun  n_ext
cod_mun periodo                                                  
02001   2021         59.0        21.0      127.0     495.0   50.0
        2022         58.0        22.0      124.0     488.0   47.0
        2023         63.0        24.0      136.0     498.0   47.0
        2024         60.0        23.0      137.0     496.0   44.0
        2025         63.0        22.0      138.0     476.0   48.0

In [58]:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.combine_first.html
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.update.html
# combine_first usará datos de padrón cuando haya de ambos, update usará de censo si overwrite=True (por defecto)
df.update(pob)
df[
    (df.index.get_level_values("periodo").isin([i for i in range(2018, 2026)]))
    & (df.index.get_level_values("cod_mun") == "02003")
].head(10)

nombre  hombres  mujeres     total    n_ext  n_dif_ca  \
cod_mun periodo                                                            
02003   2018     Albacete  84613.0  88437.0  173050.0  13072.0   17570.0   
        2019     Albacete  84687.0  88642.0  173329.0  13559.0   17567.0   
        2020     Albacete  85214.0  89122.0  174336.0  14605.0   17592.0   
        2021     Albacete  84365.0  88357.0  172722.0  14371.0   17349.0   
        2022     Albacete  84230.0  88127.0  172357.0  14371.0   17227.0   
        2023     Albacete  84585.0  88621.0  173206.0  16409.0   17237.0   
        2024     Albacete  85115.0  89022.0  174137.0  17757.0   17249.0   
        2025     Albacete  86013.0  89387.0  175400.0  19214.0   17209.0   

                 n_dif_prov  n_dif_mun  n_en_mun   ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                                
02003   2018         6562.0    27528.0  108318.0  27755.0  116050.0  29245.0   
        2019         6603.0    27166.0  108434.0  28462.0  115924.0  28943.0   
        2020         6620.0    27034.0  108485.0  29225.0  116523.0  28588.0   
        2021         6725.0    27314.0  107047.0  29251.0  115569.0  27902.0   
        2022         6699.0    26922.0  106841.0  29927.0  115037.0  27393.0   
        2023         6826.0    26405.0  106325.0      NaN       NaN      NaN   
        2024         6833.0    26068.0  106166.0      NaN       NaN      NaN   
        2025         6918.0    25794.0  105933.0      NaN       NaN      NaN   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02003   2018        1515.0     3802.0      779.0     4199.0       5317.0   
        2019        1938.0     3844.0      848.0     3781.0       5782.0   
        2020        1092.0     2815.0      666.0     3447.0       3907.0   
        2021        1570.0     3407.0     1245.0     3988.0       4977.0   
        2022           NaN        NaN        NaN        NaN          NaN   
        2023           NaN        NaN        NaN        NaN          NaN   
        2024           NaN        NaN        NaN        NaN          NaN   
        2025           NaN        NaN        NaN        NaN          NaN   

                 bajas_total  
cod_mun periodo               
02003   2018          4978.0  
        2019          4629.0  
        2020          4113.0  
        2021          5233.0  
        2022             NaN  
        2023             NaN  
        2024             NaN  
        2025             NaN

### 3.1.2. Edad

In [59]:
pob = pd.read_csv(path, sep=";", decimal=",", thousands=".")
pob = pob.loc[
    pob["Municipios"].str.startswith(("02", "13", "16", "19", "45"))
].rename(columns={"Relacion entre lugar de nacimiento y residencia": "Relación"})
# print(pob["Municipios"].str[:2].unique())
pob.head()

,Total Nacional,Provincias,Municipios,Sexo,Edad,Relación,Periodo,Total
19440,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2025,747.0
19441,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2024,760.0
19442,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2023,768.0
19443,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2022,739.0
19444,Total Nacional,02 Albacete,02001 Abengibre,Total,Todas las edades,Total,2021,752.0


In [60]:
pob["Edad"].unique()

<ArrowStringArray>
['Todas las edades', 'Menos de 16 años', 'De 16 a 64 años', '65 y más años']
Length: 4, dtype: str

In [61]:
rename = {
    "Menos de 16 años":  "ed_0_15",
    "De 16 a 64 años": "ed_16_64",
    "65 y más años":    "ed_65+",
}

pob = split_mun(pob).rename(columns={"Periodo": "periodo"})
pob = pob.loc[
    (pob["Sexo"] == "Total") & (pob["Edad"] != "Todas las edades") & (pob["Relación"] == "Total")
    , ["cod_mun", "periodo", "Edad", "Total"]
].pivot(index=["cod_mun", "periodo"], columns="Edad", values="Total").rename(columns=rename)
pob.head()

Edad             ed_65+  ed_16_64  ed_0_15
cod_mun periodo                           
02001   2021      245.0     452.0     55.0
        2022      247.0     433.0     59.0
        2023      255.0     454.0     59.0
        2024      264.0     445.0     51.0
        2025      260.0     434.0     53.0

In [62]:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.combine_first.html
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.update.html
# combine_first usará datos de padrón cuando haya de ambos, update usará de censo si overwrite=True (por defecto)
df.update(pob)
df[
    (df.index.get_level_values("periodo").isin([i for i in range(2018, 2026)]))
    & (df.index.get_level_values("cod_mun") == "02003")
].head(10)

nombre  hombres  mujeres     total    n_ext  n_dif_ca  \
cod_mun periodo                                                            
02003   2018     Albacete  84613.0  88437.0  173050.0  13072.0   17570.0   
        2019     Albacete  84687.0  88642.0  173329.0  13559.0   17567.0   
        2020     Albacete  85214.0  89122.0  174336.0  14605.0   17592.0   
        2021     Albacete  84365.0  88357.0  172722.0  14371.0   17349.0   
        2022     Albacete  84230.0  88127.0  172357.0  14371.0   17227.0   
        2023     Albacete  84585.0  88621.0  173206.0  16409.0   17237.0   
        2024     Albacete  85115.0  89022.0  174137.0  17757.0   17249.0   
        2025     Albacete  86013.0  89387.0  175400.0  19214.0   17209.0   

                 n_dif_prov  n_dif_mun  n_en_mun   ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                                
02003   2018         6562.0    27528.0  108318.0  27755.0  116050.0  29245.0   
        2019         6603.0    27166.0  108434.0  28462.0  115924.0  28943.0   
        2020         6620.0    27034.0  108485.0  29225.0  116523.0  28588.0   
        2021         6725.0    27314.0  107047.0  29216.0  115540.0  28050.0   
        2022         6699.0    26922.0  106841.0  29902.0  114666.0  27492.0   
        2023         6826.0    26405.0  106325.0  30635.0  115499.0  27068.0   
        2024         6833.0    26068.0  106166.0  31589.0  115890.0  26594.0   
        2025         6918.0    25794.0  105933.0  32705.0  116412.0  25951.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02003   2018        1515.0     3802.0      779.0     4199.0       5317.0   
        2019        1938.0     3844.0      848.0     3781.0       5782.0   
        2020        1092.0     2815.0      666.0     3447.0       3907.0   
        2021        1570.0     3407.0     1245.0     3988.0       4977.0   
        2022           NaN        NaN        NaN        NaN          NaN   
        2023           NaN        NaN        NaN        NaN          NaN   
        2024           NaN        NaN        NaN        NaN          NaN   
        2025           NaN        NaN        NaN        NaN          NaN   

                 bajas_total  
cod_mun periodo               
02003   2018          4978.0  
        2019          4629.0  
        2020          4113.0  
        2021          5233.0  
        2022             NaN  
        2023             NaN  
        2024             NaN  
        2025             NaN

## 3.2. Educación

In [63]:
path = base_path + "educacion_2021-2024/"

> SE USAN LOS VALORES `15 Y MÁS AÑOS`

### 3.2.1. Completados

In [64]:
comp_mas = pd.read_csv(path+"completados_sexo_edad_500+.csv", sep=";", decimal=",", thousands=".")
comp_mas = comp_mas.rename(columns={"Municipios de 500 habitantes o más": "Municipios"})
comp_men = pd.read_csv(path+"completados_sexo_500-.csv", sep=";", decimal=",", thousands=".")
comp_men = comp_men.rename(columns={"Municipios entre 50 y 500 habitantes": "Municipios"})
comp = pd.concat([comp_mas, comp_men], ignore_index=True)
del comp_mas; del comp_men
comp.head()

,Municipios,Sexo,Edad,Nivel de estudios,Periodo,Total
0,01001 Alegría-Dulantzi,Total,15 y más años,Total,2024,2503
1,01001 Alegría-Dulantzi,Total,15 y más años,Total,2023,2452
2,01001 Alegría-Dulantzi,Total,15 y más años,Total,2022,2428
3,01001 Alegría-Dulantzi,Total,15 y más años,Total,2021,2369
4,01001 Alegría-Dulantzi,Total,15 y más años,Educación primaria e inferior,2024,269


In [65]:
comp = comp.loc[
    (comp["Municipios"].str.startswith(("02", "13", "16", "19", "45")))
    & (comp["Sexo"] == "Total")
    & ((comp["Edad"] == "15 y más años") | (comp["Edad"].isna())) # Municipios -500 no tienen campo edad
    & (comp["Nivel de estudios"] != "Total")
, ["Municipios", "Periodo", "Nivel de estudios", "Total"]]
print(comp["Municipios"].str[:2].unique())
comp.head()

<ArrowStringArray>
['02', '13', '16', '19', '45']
Length: 5, dtype: str


,Municipios,Periodo,Nivel de estudios,Total
23764,02001 Abengibre,2024,Educación primaria e inferior,197
23765,02001 Abengibre,2023,Educación primaria e inferior,208
23766,02001 Abengibre,2022,Educación primaria e inferior,215
23767,02001 Abengibre,2021,Educación primaria e inferior,224
23768,02001 Abengibre,2024,Primera etapa de Educación Secundaria y similar,238


In [66]:
comp = split_mun(comp).rename(columns={"Periodo": "periodo"})
comp = comp.pivot(index=["cod_mun", "periodo"], columns="Nivel de estudios", values="Total")
comp.head()

Nivel de estudios  Educación primaria e inferior  Educación superior  \
cod_mun periodo                                                        
02001   2021                                 224                 150   
        2022                                 215                 149   
        2023                                 208                 173   
        2024                                 197                 176   
02002   2021                                 165                  70   

Nivel de estudios  Primera etapa de Educación Secundaria y similar  \
cod_mun periodo                                                      
02001   2021                                                   233   
        2022                                                   223   
        2023                                                   230   
        2024                                                   238   
02002   2021                                                   178   

Nivel de estudios  Segunda etapa de Educación Secundaria y Educación Postsecundaria no Superior  
cod_mun periodo                                                                                  
02001   2021                                                                                 93  
        2022                                                                                 97  
        2023                                                                                102  
        2024                                                                                100  
02002   2021                                                                                 57

In [67]:
rename = {
    "Educación primaria e inferior":                                                "edu_primaria_inf",
    "Primera etapa de Educación Secundaria y similar":                              "edu_secundaria_1",
    "Segunda etapa de Educación Secundaria y Educación Postsecundaria no Superior": "edu_secundaria_2",
    "Educación superior":                                                           "edu_superior",
}
comp = comp.rename(columns=rename)
comp.head()

Nivel de estudios  edu_primaria_inf  edu_superior  edu_secundaria_1  \
cod_mun periodo                                                       
02001   2021                    224           150               233   
        2022                    215           149               223   
        2023                    208           173               230   
        2024                    197           176               238   
02002   2021                    165            70               178   

Nivel de estudios  edu_secundaria_2  
cod_mun periodo                      
02001   2021                     93  
        2022                     97  
        2023                    102  
        2024                    100  
02002   2021                     57

In [68]:
df = df.join(comp, how="left")
df[df["edu_superior"].notna()].head()

nombre  hombres  mujeres  total  n_ext  n_dif_ca  \
cod_mun periodo                                                        
02001   2021     Abengibre    365.0    383.0  748.0   50.0      59.0   
        2022     Abengibre    363.0    376.0  739.0   47.0      58.0   
        2023     Abengibre    369.0    391.0  760.0   47.0      63.0   
        2024     Abengibre    374.0    385.0  759.0   44.0      60.0   
02002   2021        Alatoz    272.0    224.0  496.0   58.0      42.0   

                 n_dif_prov  n_dif_mun  n_en_mun  ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                               
02001   2021           21.0      127.0     495.0   245.0     452.0     55.0   
        2022           22.0      124.0     488.0   247.0     433.0     59.0   
        2023           24.0      136.0     498.0   255.0     454.0     59.0   
        2024           23.0      137.0     496.0   264.0     445.0     51.0   
02002   2021            4.0      117.0     289.0   184.0     282.0     44.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2021           1.0       18.0        0.0       23.0         19.0   
        2022           NaN        NaN        NaN        NaN          NaN   
        2023           NaN        NaN        NaN        NaN          NaN   
        2024           NaN        NaN        NaN        NaN          NaN   
02002   2021           2.0       27.0        1.0       27.0         29.0   

                 bajas_total  edu_primaria_inf  edu_superior  \
cod_mun periodo                                                
02001   2021            23.0             224.0         150.0   
        2022             NaN             215.0         149.0   
        2023             NaN             208.0         173.0   
        2024             NaN             197.0         176.0   
02002   2021            28.0             165.0          70.0   

                 edu_secundaria_1  edu_secundaria_2  
cod_mun periodo                                      
02001   2021                233.0              93.0  
        2022                223.0              97.0  
        2023                230.0             102.0  
        2024                238.0             100.0  
02002   2021                178.0              57.0

### 3.2.2. En curso y relación con la actividad

In [69]:
act_mas = pd.read_csv(path+"actividad_sexo_edad_500+.csv", sep=";", decimal=",", thousands=".")
act_mas = act_mas.rename(columns={"Municipios de 500 habitantes o más": "Municipios"})
act_men = pd.read_csv(path+"actividad_sexo_500-.csv", sep=";", decimal=",", thousands=".")
act_men = act_men.rename(columns={"Municipios entre 50 y 500 habitantes": "Municipios"})
act = pd.concat([act_mas, act_men], ignore_index=True)
del act_mas; del act_men
act.head()

,Municipios,Sexo,Edad,Relación con la actividad,Periodo,Total
0,01001 Alegría-Dulantzi,Total,16 y más años,Total,2024,2445
1,01001 Alegría-Dulantzi,Total,16 y más años,Total,2023,2403
2,01001 Alegría-Dulantzi,Total,16 y más años,Total,2022,2377
3,01001 Alegría-Dulantzi,Total,16 y más años,Total,2021,2331
4,01001 Alegría-Dulantzi,Total,16 y más años,Ocupado/a,2024,1437


In [70]:
act = act.loc[
    (act["Municipios"].str.startswith(("02", "13", "16", "19", "45")))
    & (act["Sexo"] == "Total")
    & ((act["Edad"] == "16 y más años") | (act["Edad"].isna())) # Municipios -500 no tienen campo edad
    & (act["Relación con la actividad"] != "Total")
, ["Municipios", "Periodo", "Relación con la actividad", "Total"]]
# print(act["Municipios"].str[:2].unique())
act.head()

,Municipios,Periodo,Relación con la actividad,Total
20596,02001 Abengibre,2024,Ocupado/a,327
20597,02001 Abengibre,2023,Ocupado/a,326
20598,02001 Abengibre,2022,Ocupado/a,311
20599,02001 Abengibre,2021,Ocupado/a,305
20600,02001 Abengibre,2024,Parado/a,25


In [71]:
act = split_mun(act).rename(columns={"Periodo": "periodo"})
act = act.pivot(index=["cod_mun", "periodo"], columns="Relación con la actividad", values="Total")
act.head()

Relación con la actividad  Inactivo/a  Ocupado/a  Parado/a
cod_mun periodo                                           
02001   2021                      345        305        47
        2022                      344        311        25
        2023                      351        326        32
        2024                      357        327        25
02002   2021                      241        188        37

In [72]:
rename = {
    "Inactivo/a":   "act_inactivo",
    "Ocupado/a":    "act_ocupado",
    "Parado/a":     "act_parado",
}
act = act.rename(columns=rename)
act.head()

Relación con la actividad  act_inactivo  act_ocupado  act_parado
cod_mun periodo                                                 
02001   2021                        345          305          47
        2022                        344          311          25
        2023                        351          326          32
        2024                        357          327          25
02002   2021                        241          188          37

In [73]:
df = df.join(act, how="left")
df[df["act_inactivo"].notna()].head()

nombre  hombres  mujeres  total  n_ext  n_dif_ca  \
cod_mun periodo                                                        
02001   2021     Abengibre    365.0    383.0  748.0   50.0      59.0   
        2022     Abengibre    363.0    376.0  739.0   47.0      58.0   
        2023     Abengibre    369.0    391.0  760.0   47.0      63.0   
        2024     Abengibre    374.0    385.0  759.0   44.0      60.0   
02002   2021        Alatoz    272.0    224.0  496.0   58.0      42.0   

                 n_dif_prov  n_dif_mun  n_en_mun  ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                               
02001   2021           21.0      127.0     495.0   245.0     452.0     55.0   
        2022           22.0      124.0     488.0   247.0     433.0     59.0   
        2023           24.0      136.0     498.0   255.0     454.0     59.0   
        2024           23.0      137.0     496.0   264.0     445.0     51.0   
02002   2021            4.0      117.0     289.0   184.0     282.0     44.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2021           1.0       18.0        0.0       23.0         19.0   
        2022           NaN        NaN        NaN        NaN          NaN   
        2023           NaN        NaN        NaN        NaN          NaN   
        2024           NaN        NaN        NaN        NaN          NaN   
02002   2021           2.0       27.0        1.0       27.0         29.0   

                 bajas_total  edu_primaria_inf  edu_superior  \
cod_mun periodo                                                
02001   2021            23.0             224.0         150.0   
        2022             NaN             215.0         149.0   
        2023             NaN             208.0         173.0   
        2024             NaN             197.0         176.0   
02002   2021            28.0             165.0          70.0   

                 edu_secundaria_1  edu_secundaria_2  act_inactivo  \
cod_mun periodo                                                     
02001   2021                233.0              93.0         345.0   
        2022                223.0              97.0         344.0   
        2023                230.0             102.0         351.0   
        2024                238.0             100.0         357.0   
02002   2021                178.0              57.0         241.0   

                 act_ocupado  act_parado  
cod_mun periodo                           
02001   2021           305.0        47.0  
        2022           311.0        25.0  
        2023           326.0        32.0  
        2024           327.0        25.0  
02002   2021           188.0        37.0

## 3.3. Ocupación

In [74]:
path = base_path + "ocupacion_2021-2023/"

### 3.3.1. Ocupación

In [75]:
# ocu_mas = pd.read_csv(path+"ocupacion_sexo_edad_500+.csv", sep=";", decimal=",", thousands=".")
# ocu_mas = ocu_mas.rename(columns={"Municipios de 500 habitantes o más": "Municipios"})
# ocu_min = pd.read_csv(path+"ocupacion_sexo_500-.csv", sep=";", decimal=",", thousands=".")
# ocu_min = ocu_min.rename(columns={"Municipios entre 50 y 500 habitantes": "Municipios"})
# ocu = pd.concat([ocu_mas, ocu_min], ignore_index=True)
# ocu.head()

In [76]:
# ocu = ocu.loc[
#     (ocu["Municipios"].str.startswith(("02", "13", "16", "19", "45")))
#     & (ocu["Sexo"] == "Total")
#     & ((ocu["Edad"] == "16 y más años") | (ocu["Edad"].isna())) # Municipios -500 no tienen campo edad
#     & (ocu["Ocupación"] != "Total")
# , ["Municipios", "Periodo", "Ocupación", "Total"]]
# print(ocu["Municipios"].str[:2].unique())
# ocu.head()

In [77]:
# ocu.loc[ocu.isna().any(axis=1), ["Municipios"]].value_counts()

In [78]:
# ocu[ocu["Municipios"] == "02013 Balsa de Ves"]

In [79]:
# ocu = ocu.pivot(index=["Municipios", "Periodo"], columns="Ocupación", values="Total")
# ocu.head()

In [80]:
# rename = {
#     "No consta": "ocu_no_consta",
#     "Ocupaciones elementales": "ocu_elemental",
#     "Trabajadores cualificados y oficiales/operarios de nivel bajo": "ocu_nivel_bajo",
#     "Directores/gerentes y profesionales/técnicos de nivel medio o alto": "ocu_nivel_medio_alto"
# }
# ocu = ocu.rename(columns=rename)
# ocu.head()

In [81]:
# df = df.join(ocu, how="left")
# df[df["ocu_nivel_medio_alto"].notna()].head()

### 3.3.2. Situación profesional

In [82]:
sit_mas = pd.read_csv(path+"situacion_profesional_sexo_edad_500+.csv", sep=";", decimal=",", thousands=".")
sit_mas = sit_mas.rename(columns={"Municipios de 500 habitantes o más": "Municipios"})
sit_min = pd.read_csv(path+"situacion_profesional_sexo_500-.csv", sep=";", decimal=",", thousands=".")
sit_min = sit_min.rename(columns={"Municipios entre 50 y 500 habitantes": "Municipios"})
sit = pd.concat([sit_mas, sit_min], ignore_index=True)
sit.head()

,Municipios,Edad,Sexo,Situación profesional,Periodo,Total
0,01001 Alegría-Dulantzi,16 y más años,Total,Total,2023,1431
1,01001 Alegría-Dulantzi,16 y más años,Total,Total,2022,1400
2,01001 Alegría-Dulantzi,16 y más años,Total,Total,2021,1339
3,01001 Alegría-Dulantzi,16 y más años,Total,Trabajador por cuenta propia,2023,200
4,01001 Alegría-Dulantzi,16 y más años,Total,Trabajador por cuenta propia,2022,203


In [83]:
sit["Situación profesional"].unique()

<ArrowStringArray>
[                                       'Total',
                 'Trabajador por cuenta propia',
 'Trabajador por cuenta ajena y otra situación']
Length: 3, dtype: str

In [84]:
sit = sit.loc[
    (sit["Municipios"].str.startswith(("02", "13", "16", "19", "45")))
    & (sit["Sexo"] == "Total")
    & ((sit["Edad"] == "16 y más años") | (sit["Edad"].isna()))
    & (sit["Situación profesional"] != "Total")
, ["Municipios", "Periodo", "Situación profesional", "Total"]]
sit.head()

,Municipios,Periodo,Situación profesional,Total
11586,02001 Abengibre,2023,Trabajador por cuenta propia,86
11587,02001 Abengibre,2022,Trabajador por cuenta propia,81
11588,02001 Abengibre,2021,Trabajador por cuenta propia,86
11589,02001 Abengibre,2023,Trabajador por cuenta ajena y otra situación,240
11590,02001 Abengibre,2022,Trabajador por cuenta ajena y otra situación,230


In [85]:
sit.loc[sit.isna().any(axis=1), ["Municipios"]].value_counts()

Series([], Name: count, dtype: int64)

In [86]:
rename = {
    "Trabajador por cuenta propia": "sp_cuenta_propia",
    "Trabajador por cuenta ajena y otra situación": "sp_cuenta_ajena_y_otros",
}

sit = split_mun(sit).rename(columns={"Periodo": "periodo"})
sit = sit.pivot(index=["cod_mun", "periodo"], columns="Situación profesional", values="Total") \
.rename(columns=rename)
sit.head()

Situación profesional  sp_cuenta_ajena_y_otros  sp_cuenta_propia
cod_mun periodo                                                 
02001   2021                               219                86
        2022                               230                81
        2023                               240                86
02002   2021                               122                66
        2022                               125                64

In [87]:
df = df.join(sit, how="left")
df[df["sp_cuenta_propia"].notna()].head()

nombre  hombres  mujeres  total  n_ext  n_dif_ca  \
cod_mun periodo                                                        
02001   2021     Abengibre    365.0    383.0  748.0   50.0      59.0   
        2022     Abengibre    363.0    376.0  739.0   47.0      58.0   
        2023     Abengibre    369.0    391.0  760.0   47.0      63.0   
02002   2021        Alatoz    272.0    224.0  496.0   58.0      42.0   
        2022        Alatoz    276.0    220.0  496.0   53.0      49.0   

                 n_dif_prov  n_dif_mun  n_en_mun  ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                               
02001   2021           21.0      127.0     495.0   245.0     452.0     55.0   
        2022           22.0      124.0     488.0   247.0     433.0     59.0   
        2023           24.0      136.0     498.0   255.0     454.0     59.0   
02002   2021            4.0      117.0     289.0   184.0     282.0     44.0   
        2022            3.0      121.0     276.0   179.0     285.0     38.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2021           1.0       18.0        0.0       23.0         19.0   
        2022           NaN        NaN        NaN        NaN          NaN   
        2023           NaN        NaN        NaN        NaN          NaN   
02002   2021           2.0       27.0        1.0       27.0         29.0   
        2022           NaN        NaN        NaN        NaN          NaN   

                 bajas_total  edu_primaria_inf  edu_superior  \
cod_mun periodo                                                
02001   2021            23.0             224.0         150.0   
        2022             NaN             215.0         149.0   
        2023             NaN             208.0         173.0   
02002   2021            28.0             165.0          70.0   
        2022             NaN             147.0          76.0   

                 edu_secundaria_1  edu_secundaria_2  act_inactivo  \
cod_mun periodo                                                     
02001   2021                233.0              93.0         345.0   
        2022                223.0              97.0         344.0   
        2023                230.0             102.0         351.0   
02002   2021                178.0              57.0         241.0   
        2022                182.0              62.0         244.0   

                 act_ocupado  act_parado  sp_cuenta_ajena_y_otros  \
cod_mun periodo                                                     
02001   2021           305.0        47.0                    219.0   
        2022           311.0        25.0                    230.0   
        2023           326.0        32.0                    240.0   
02002   2021           188.0        37.0                    122.0   
        2022           189.0        31.0                    125.0   

                 sp_cuenta_propia  
cod_mun periodo                    
02001   2021                 86.0  
        2022                 81.0  
        2023                 86.0  
02002   2021                 66.0  
        2022                 64.0

### 3.3.3. Rama de actividad económica

In [88]:
act_mas = pd.read_csv(path+"rama_actividad_sexo_edad_500+.csv", sep=";", decimal=",", thousands=".")
act_mas = act_mas.rename(columns={"Municipios de 500 habitantes o más": "Municipios"})
act_min = pd.read_csv(path+"rama_actividad_sexo_500-.csv", sep=";", decimal=",", thousands=".")
act_min = act_min.rename(columns={"Municipios entre 50 y 500 habitantes": "Municipios"})
act = pd.concat([act_mas, act_min], ignore_index=True)
del act_mas; del act_min
act.head()

,Municipios,Edad,Sexo,Actividad económica,Periodo,Total
0,01001 Alegría-Dulantzi,16 y más años,Total,Total CNAE,2023,1431
1,01001 Alegría-Dulantzi,16 y más años,Total,Total CNAE,2022,1400
2,01001 Alegría-Dulantzi,16 y más años,Total,Total CNAE,2021,1339
3,01001 Alegría-Dulantzi,16 y más años,Total,"Agricultura, ganadería y pesca",2023,20
4,01001 Alegría-Dulantzi,16 y más años,Total,"Agricultura, ganadería y pesca",2022,20


In [89]:
act["Actividad económica"].unique()

<ArrowStringArray>
[                    'Total CNAE', 'Agricultura, ganadería y pesca',
                      'Industria',                   'Construcción',
                      'Servicios',                      'No consta']
Length: 6, dtype: str

In [90]:
act = act.loc[
    (act["Municipios"].str.startswith(("02", "13", "16", "19", "45")))
    & (act["Sexo"] == "Total")
    & ((act["Edad"] == "16 y más años") | (act["Edad"].isna()))
    & (act["Actividad económica"] != "Total CNAE")
, ["Municipios", "Periodo", "Actividad económica", "Total"]]
act.head()

,Municipios,Periodo,Actividad económica,Total
23169,02001 Abengibre,2023,"Agricultura, ganadería y pesca",33
23170,02001 Abengibre,2022,"Agricultura, ganadería y pesca",36
23171,02001 Abengibre,2021,"Agricultura, ganadería y pesca",29
23172,02001 Abengibre,2023,Industria,57
23173,02001 Abengibre,2022,Industria,48


In [91]:
act.loc[act.isna().any(axis=1), ["Municipios"]].value_counts()

Series([], Name: count, dtype: int64)

In [92]:
rename = {
    "Agricultura, ganadería y pesca": "ae_agricultura_ganaderia_pesca",
    "Industria": "ae_industria",
    "Construcción": "ae_construccion",
    "Servicios": "ae_servicios",
    "No consta": "ae_no_consta"
}

act = split_mun(act).rename(columns={"Periodo": "periodo"})
act = act.pivot(index=["cod_mun", "periodo"], columns="Actividad económica", values="Total") \
.rename(columns=rename)
act.head()

Actividad económica  ae_agricultura_ganaderia_pesca  ae_construccion  \
cod_mun periodo                                                        
02001   2021                                     29               11   
        2022                                     36               11   
        2023                                     33               11   
02002   2021                                     47               27   
        2022                                     44               27   

Actividad económica  ae_industria  ae_no_consta  ae_servicios  
cod_mun periodo                                                
02001   2021                   57             1           207  
        2022                   48             1           215  
        2023                   57             1           224  
02002   2021                   24             0            90  
        2022                   23             0            95

In [93]:
df = df.join(act, how="left")
df[df["ae_servicios"].notna()].head()

nombre  hombres  mujeres  total  n_ext  n_dif_ca  \
cod_mun periodo                                                        
02001   2021     Abengibre    365.0    383.0  748.0   50.0      59.0   
        2022     Abengibre    363.0    376.0  739.0   47.0      58.0   
        2023     Abengibre    369.0    391.0  760.0   47.0      63.0   
02002   2021        Alatoz    272.0    224.0  496.0   58.0      42.0   
        2022        Alatoz    276.0    220.0  496.0   53.0      49.0   

                 n_dif_prov  n_dif_mun  n_en_mun  ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                               
02001   2021           21.0      127.0     495.0   245.0     452.0     55.0   
        2022           22.0      124.0     488.0   247.0     433.0     59.0   
        2023           24.0      136.0     498.0   255.0     454.0     59.0   
02002   2021            4.0      117.0     289.0   184.0     282.0     44.0   
        2022            3.0      121.0     276.0   179.0     285.0     38.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2021           1.0       18.0        0.0       23.0         19.0   
        2022           NaN        NaN        NaN        NaN          NaN   
        2023           NaN        NaN        NaN        NaN          NaN   
02002   2021           2.0       27.0        1.0       27.0         29.0   
        2022           NaN        NaN        NaN        NaN          NaN   

                 bajas_total  edu_primaria_inf  edu_superior  \
cod_mun periodo                                                
02001   2021            23.0             224.0         150.0   
        2022             NaN             215.0         149.0   
        2023             NaN             208.0         173.0   
02002   2021            28.0             165.0          70.0   
        2022             NaN             147.0          76.0   

                 edu_secundaria_1  edu_secundaria_2  act_inactivo  \
cod_mun periodo                                                     
02001   2021                233.0              93.0         345.0   
        2022                223.0              97.0         344.0   
        2023                230.0             102.0         351.0   
02002   2021                178.0              57.0         241.0   
        2022                182.0              62.0         244.0   

                 act_ocupado  act_parado  sp_cuenta_ajena_y_otros  \
cod_mun periodo                                                     
02001   2021           305.0        47.0                    219.0   
        2022           311.0        25.0                    230.0   
        2023           326.0        32.0                    240.0   
02002   2021           188.0        37.0                    122.0   
        2022           189.0        31.0                    125.0   

                 sp_cuenta_propia  ae_agricultura_ganaderia_pesca  \
cod_mun periodo                                                     
02001   2021                 86.0                            29.0   
        2022                 81.0                            36.0   
        2023                 86.0                            33.0   
02002   2021                 66.0                            47.0   
        2022                 64.0                            44.0   

                 ae_construccion  ae_industria  ae_no_consta  ae_servicios  
cod_mun periodo                                                             
02001   2021                11.0          57.0           1.0         207.0  
        2022                11.0          48.0           1.0         215.0  
        2023                11.0          57.0           1.0         224.0  
02002   2021                27.0          24.0           0.0          90.0  
        2022                27.0          23.0           0.0     

# 4. Fenómenos demográficos

In [94]:
base_path = "datos/fenomenos_demograficos/"

## 4.1. Migraciones y cambios de residencia

> **DATOS DE CADA AÑO NATURAL**

> 2021-2024, continúa datos de altas/bajas de padrón

In [95]:
path = base_path + "migraciones_residencia/"

In [96]:
b_ext = pd.read_csv(path+"emigraciones_ext.csv", sep=";", decimal=",", thousands=".").rename(columns={"Total": "bajas_ext"})
b_int = pd.read_csv(path+"emigraciones_int.csv", sep=";", decimal=",", thousands=".").rename(columns={"Total": "bajas_int"})
a_ext = pd.read_csv(path+"inmigraciones_ext.csv", sep=";", decimal=",", thousands=".").rename(columns={"Total": "altas_ext"})
a_int = pd.read_csv(path+"inmigraciones_int.csv", sep=";", decimal=",", thousands=".").rename(columns={"Total": "altas_int"})

In [97]:
b_int.head()

,Total Nacional,Provincias,Municipios,Nacionalidad,Sexo,Periodo,bajas_int
0,Total Nacional,NaN,NaN,Total,Ambos sexos,2024,1754162.0
1,Total Nacional,NaN,NaN,Total,Ambos sexos,2023,1719908.0
2,Total Nacional,NaN,NaN,Total,Ambos sexos,2022,1725546.0
3,Total Nacional,NaN,NaN,Total,Ambos sexos,2021,1670717.0
4,Total Nacional,NaN,NaN,Total,Hombres,2024,878334.0


In [98]:
def prepare_df(df):
    df = df.copy()
    df["Municipios"] = df["Municipios"].fillna(df["Provincias"])
    df = df.loc[
        (df["Municipios"].str.startswith(("02", "13", "16", "19", "45")))
        & (df["Sexo"] == "Ambos sexos")
    ].drop(columns=["Total Nacional", "Provincias", "Sexo"])
    if "Nacionalidad" in df.columns:
        df = df.loc[df["Nacionalidad"] == "Total"]
        df = df.drop(columns="Nacionalidad")
    df = split_mun(df).rename(columns={"Periodo": "periodo"}).drop(columns="nombre")
    df = df.set_index(["cod_mun", "periodo"])
    return df

prepare_df(b_int).head()

bajas_int
cod_mun periodo           
02      2024       10772.0
        2023       10979.0
        2022       11199.0
        2021        9844.0
02001   2024          39.0

> !!! NOTA: En 2021 se solapan datos de padrón y de censo, y se tienen cifras ligeramente distintas, se usarán los datos de censo sobre los de padrón

In [99]:
# print(f"BAJAS INTERIORES DEL PADRÓN EN ABENGIBRE: {prepare_df(b_int).loc[("02001", 2021), "bajas_int"]}")
# print(f"BAJAS INTERIORES DEL CENSO EN ABENGIBRE: {df.loc[("02001", 2021), "bajas_int"]}")
# print()
# print(f"BAJAS INTERIORES DEL PADRÓN EN ALBACETE: {prepare_df(b_int).loc[("02003", 2021), "bajas_int"]}")
# print(f"BAJAS INTERIORES DEL CENSO EN ALBACETE: {df.loc[("02003", 2021), "bajas_int"]}")

In [100]:
b_ext = prepare_df(b_ext)
b_int = prepare_df(b_int)
a_ext = prepare_df(a_ext)
a_int = prepare_df(a_int)

In [101]:
print(b_ext.isna().sum())
print(b_int.isna().sum())
print(a_ext.isna().sum())
print(a_int.isna().sum())

bajas_ext    0
dtype: int64
bajas_int    0
dtype: int64
altas_ext    0
dtype: int64
altas_int    0
dtype: int64


In [102]:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.update.html#pandas.DataFrame.update
df.update(b_ext, overwrite=True)
df.update(b_int, overwrite=True)
df.update(a_ext, overwrite=True)
df.update(a_int, overwrite=True)
del b_ext; del b_int; del a_ext; del a_int

In [103]:
df[(df["bajas_ext"].notna()) & (df.index.get_level_values("periodo") >= 2020) & (df.index.get_level_values("cod_mun") != "02")][["bajas_int", "bajas_ext", "altas_int", "altas_ext"]].head()

bajas_int  bajas_ext  altas_int  altas_ext
cod_mun periodo                                            
02001   2020          21.0        0.0       17.0        1.0
        2021          22.0        6.0       17.0        3.0
        2022          21.0        2.0       56.0        5.0
        2023          50.0        7.0       57.0        1.0
        2024          39.0        4.0       34.0        4.0

## 4.2. Movimiento nartural

> **DATOS DE 1 DE CADA AÑO NATURAL**

In [104]:
path = base_path + "movimiento_natural/"

In [105]:
files = glob.glob(path+"fenomenos_*_*.csv")
temp = [pd.read_csv(f, sep=";", decimal=",", thousands=".").assign(Periodo=int(re.search(r"fenomenos_[a-z]{2}_(\d{4})\.csv$", f).group(1))) for f in files]
mov = pd.concat(temp, ignore_index=True)
del temp
mov.head()

,Municipios,Fenómeno demográfico,Total,Periodo
0,13001 Abenójar,nacidos vivos por residencia materna,8,2016
1,13001 Abenójar,muertes fetales tardías por residencia materna,0,2016
2,13001 Abenójar,matrimonios por el lugar en que han fijado residencia,1,2016
3,13001 Abenójar,fallecidos por el lugar de residencia,22,2016
4,13001 Abenójar,crecimiento vegetativo,-14,2016


In [106]:
mov = split_mun(mov).rename(columns={"Periodo": "periodo"})
mov = mov.pivot(index=["cod_mun", "periodo"], columns=["Fenómeno demográfico"], values="Total")
mov.head()

Fenómeno demográfico  crecimiento vegetativo  \
cod_mun periodo                                
02001   2015                             -10   
        2016                             -13   
        2017                             -15   
        2018                             -10   
        2019                              -6   

Fenómeno demográfico  fallecidos por el lugar de residencia  \
cod_mun periodo                                               
02001   2015                                             12   
        2016                                             16   
        2017                                             17   
        2018                                             11   
        2019                                             12   

Fenómeno demográfico  matrimonios por el lugar en que han fijado residencia  \
cod_mun periodo                                                               
02001   2015                                                              0   
        2016                                                              1   
        2017                                                              2   
        2018                                                              3   
        2019                                                              1   

Fenómeno demográfico  muertes fetales tardías por residencia materna  \
cod_mun periodo                                                        
02001   2015                                                       0   
        2016                                                       0   
        2017                                                       0   
        2018                                                       0   
        2019                                                       0   

Fenómeno demográfico  nacidos vivos por residencia materna  
cod_mun periodo                                             
02001   2015                                             2  
        2016                                             3  
        2017                                             2  
        2018                                             1  
        2019                                             6

In [107]:
mov.isna().sum()

Fenómeno demográfico
crecimiento vegetativo                                   0
fallecidos por el lugar de residencia                    0
matrimonios por el lugar en que han fijado residencia    0
muertes fetales tardías por residencia materna           0
nacidos vivos por residencia materna                     0
dtype: int64

In [108]:
rename = {
    "fallecidos por el lugar de residencia":                    "mnp_fallecidos",
    "matrimonios por el lugar en que han fijado residencia":    "mnp_matrimonios",
    "nacidos vivos por residencia materna":                     "mnp_nacimientos",
    "crecimiento vegetativo":                                   "mnp_crecimiento_veg",
}
mov = mov.rename(columns=rename).drop(columns="muertes fetales tardías por residencia materna")
mov.head()

Fenómeno demográfico  mnp_crecimiento_veg  mnp_fallecidos  mnp_matrimonios  \
cod_mun periodo                                                              
02001   2015                          -10              12                0   
        2016                          -13              16                1   
        2017                          -15              17                2   
        2018                          -10              11                3   
        2019                           -6              12                1   

Fenómeno demográfico  mnp_nacimientos  
cod_mun periodo                        
02001   2015                        2  
        2016                        3  
        2017                        2  
        2018                        1  
        2019                        6

In [109]:
df = df.join(mov, how="left")
df[df["mnp_nacimientos"].notna()].head()

nombre  hombres  mujeres  total  n_ext  n_dif_ca  \
cod_mun periodo                                                        
02001   2015     Abengibre    386.0    396.0  782.0   64.0      42.0   
        2016     Abengibre    366.0    395.0  761.0   49.0      47.0   
        2017     Abengibre    359.0    389.0  748.0   50.0      49.0   
        2018     Abengibre    363.0    394.0  757.0   55.0      53.0   
        2019     Abengibre    379.0    411.0  790.0   57.0      56.0   

                 n_dif_prov  n_dif_mun  n_en_mun  ed_65+  ed_16_64  ed_0_15  \
cod_mun periodo                                                               
02001   2015           22.0      117.0     537.0   232.0     499.0     51.0   
        2016           24.0      116.0     525.0   233.0     478.0     50.0   
        2017           21.0      111.0     517.0   225.0     479.0     44.0   
        2018           21.0      115.0     513.0   224.0     485.0     48.0   
        2019           22.0      127.0     528.0   231.0     507.0     52.0   

                 altas_ext  altas_int  bajas_ext  bajas_int  altas_total  \
cod_mun periodo                                                            
02001   2015           4.0       22.0        2.0       18.0         26.0   
        2016           4.0       30.0       26.0       30.0         34.0   
        2017           5.0       30.0        1.0       13.0         35.0   
        2018           6.0       59.0        1.0       22.0         65.0   
        2019           3.0       19.0        0.0       43.0         22.0   

                 bajas_total  edu_primaria_inf  edu_superior  \
cod_mun periodo                                                
02001   2015            20.0               NaN           NaN   
        2016            56.0               NaN           NaN   
        2017            14.0               NaN           NaN   
        2018            23.0               NaN           NaN   
        2019            43.0               NaN           NaN   

                 edu_secundaria_1  edu_secundaria_2  act_inactivo  \
cod_mun periodo                                                     
02001   2015                  NaN               NaN           NaN   
        2016                  NaN               NaN           NaN   
        2017                  NaN               NaN           NaN   
        2018                  NaN               NaN           NaN   
        2019                  NaN               NaN           NaN   

                 act_ocupado  act_parado  sp_cuenta_ajena_y_otros  \
cod_mun periodo                                                     
02001   2015             NaN         NaN                      NaN   
        2016             NaN         NaN                      NaN   
        2017             NaN         NaN                      NaN   
        2018             NaN         NaN                      NaN   
        2019             NaN         NaN                      NaN   

                 sp_cuenta_propia  ae_agricultura_ganaderia_pesca  \
cod_mun periodo                                                     
02001   2015                  NaN                             NaN   
        2016                  NaN                             NaN   
        2017                  NaN                             NaN   
        2018                  NaN                             NaN   
        2019                  NaN                             NaN   

                 ae_construccion  ae_industria  ae_no_consta  ae_servicios  \
cod_mun periodo                                                              
02001   2015                 NaN           NaN           NaN           NaN   
        2016                 NaN           NaN           NaN           NaN   
        2017                 NaN           NaN           NaN           NaN   
        2018                 NaN           NaN           NaN           NaN   
        2019                 NaN           NaN           Na

# 5. Calculos a nivel provincia

## 5.0. Acotar a 2000-2025 y crear provincias

In [110]:
df = df.reset_index()
conds = [
    df["cod_mun"].str.startswith("02"),
    df["cod_mun"].str.startswith("13"),
    df["cod_mun"].str.startswith("16"),
    df["cod_mun"].str.startswith("19"),
    df["cod_mun"].str.startswith("45"),
] 
opts = [
    "02",
    "13",
    "16",
    "19",
    "45",
]

df["cod_prov"] = np.select(conds, opts, default="N/A")

In [111]:
df = df.drop(columns=["altas_total", "bajas_total"])

In [112]:
df = df[df["periodo"] >= 2000]
df.head()

,cod_mun,periodo,nombre,hombres,mujeres,total,n_ext,n_dif_ca,n_dif_prov,n_dif_mun,n_en_mun,ed_65+,ed_16_64,ed_0_15,altas_ext,altas_int,bajas_ext,bajas_int,edu_primaria_inf,edu_superior,edu_secundaria_1,edu_secundaria_2,act_inactivo,act_ocupado,act_parado,sp_cuenta_ajena_y_otros,sp_cuenta_propia,ae_agricultura_ganaderia_pesca,ae_construccion,ae_industria,ae_no_consta,ae_servicios,mnp_crecimiento_veg,mnp_fallecidos,mnp_matrimonios,mnp_nacimientos,cod_prov
4,02,2000,Albacete,180311.0,182952.0,363263.0,4229.0,33134.0,9017.0,72544.0,244339.0,64596.0,232715.0,65952.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
5,02,2001,Albacete,182826.0,184457.0,367283.0,7560.0,33336.0,9113.0,74643.0,242631.0,65957.0,236246.0,65080.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
6,02,2002,Albacete,185458.0,186329.0,371787.0,11968.0,33561.0,9160.0,74192.0,242906.0,67020.0,240043.0,64724.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
7,02,2003,Albacete,188139.0,188417.0,376556.0,15777.0,33995.0,9348.0,73829.0,243607.0,67861.0,243724.0,64971.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
8,02,2004,Albacete,189743.0,189705.0,379448.0,18509.0,34283.0,9451.0,73507.0,243698.0,68315.0,246642.0,64491.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02


In [113]:
df.info()

<class 'pandas.DataFrame'>
Index: 24024 entries, 4 to 27719
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   cod_mun                         24024 non-null  str    
 1   periodo                         24024 non-null  int64  
 2   nombre                          24024 non-null  str    
 3   hombres                         24023 non-null  float64
 4   mujeres                         24023 non-null  float64
 5   total                           24023 non-null  float64
 6   n_ext                           23987 non-null  float64
 7   n_dif_ca                        23987 non-null  float64
 8   n_dif_prov                      23987 non-null  float64
 9   n_dif_mun                       23987 non-null  float64
 10  n_en_mun                        23987 non-null  float64
 11  ed_65+                          23987 non-null  float64
 12  ed_16_64                        23987 non-null  

In [114]:
df[df["periodo"] >= 2015].head(10)

,cod_mun,periodo,nombre,hombres,mujeres,total,n_ext,n_dif_ca,n_dif_prov,n_dif_mun,n_en_mun,ed_65+,ed_16_64,ed_0_15,altas_ext,altas_int,bajas_ext,bajas_int,edu_primaria_inf,edu_superior,edu_secundaria_1,edu_secundaria_2,act_inactivo,act_ocupado,act_parado,sp_cuenta_ajena_y_otros,sp_cuenta_propia,ae_agricultura_ganaderia_pesca,ae_construccion,ae_industria,ae_no_consta,ae_servicios,mnp_crecimiento_veg,mnp_fallecidos,mnp_matrimonios,mnp_nacimientos,cod_prov
19,02,2015,Albacete,197014.0,197566.0,394580.0,31493.0,36519.0,10241.0,68409.0,247918.0,72920.0,258810.0,62850.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
20,02,2016,Albacete,195841.0,196277.0,392118.0,30692.0,36252.0,10236.0,67541.0,247397.0,73302.0,256947.0,61869.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
21,02,2017,Albacete,194743.0,195289.0,390032.0,30319.0,36047.0,10262.0,66855.0,246549.0,73511.0,255674.0,60847.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
22,02,2018,Albacete,194158.0,194628.0,388786.0,30959.0,35953.0,10278.0,65941.0,245655.0,74080.0,254450.0,60256.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
23,02,2019,Albacete,193911.0,194256.0,388167.0,32002.0,35935.0,10335.0,65231.0,244664.0,74794.0,253910.0,59463.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
24,02,2020,Albacete,194081.0,194189.0,388270.0,34018.0,35959.0,10369.0,64606.0,243318.0,75379.0,254323.0,58568.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
25,02,2021,Albacete,193205.0,193259.0,386464.0,34662.0,36080.0,10330.0,63947.0,241445.0,75119.0,253794.0,57551.0,4542.0,9301.0,3740.0,9844.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
26,02,2022,Albacete,192963.0,192764.0,385727.0,35648.0,36176.0,10335.0,63426.0,240142.0,75870.0,253102.0,56755.0,6622.0,10422.0,2927.0,11199.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
27,02,2023,Albacete,193676.0,193498.0,387174.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6624.0,10480.0,3414.0,10979.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
28,02,2024,Albacete,194489.0,194110.0,388599.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7027.0,10211.0,3677.0,10772.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02


## 5.1. Variables que no aparecen a nivel provincia

In [115]:
var_list = [
    "n_ext", "n_dif_ca", "n_dif_prov", "n_dif_mun", "n_en_mun",
    "ed_65+", "ed_16_64", "ed_0_15",
    "altas_int", "altas_ext", "bajas_int", "bajas_ext",
    "edu_primaria_inf", "edu_superior", "edu_secundaria_1", "edu_secundaria_2",
    "act_inactivo", "act_ocupado", "act_parado",
    "sp_cuenta_propia", "sp_cuenta_ajena_y_otros",
    "ae_agricultura_ganaderia_pesca", "ae_construccion", "ae_industria", "ae_no_consta", "ae_servicios",
    "mnp_crecimiento_veg", "mnp_fallecidos", "mnp_matrimonios", "mnp_nacimientos",
]



vars_provincias = df.loc[~(df["cod_mun"].isin(PRE_CLM))].groupby(["cod_prov", "periodo"])[var_list].sum(min_count=1)
vars_provincias = vars_provincias.reset_index().rename(columns={"cod_prov": "cod_mun"}).set_index(["cod_mun", "periodo"])
vars_provincias[vars_provincias.index.get_level_values("periodo") >= 2015].head(11)

n_ext  n_dif_ca  n_dif_prov  n_dif_mun  n_en_mun   ed_65+  \
cod_mun periodo                                                                
02      2015     31493.0   36519.0     10241.0    68409.0  247918.0  72920.0   
        2016     30692.0   36252.0     10236.0    67541.0  247397.0  73302.0   
        2017     30319.0   36047.0     10262.0    66855.0  246549.0  73511.0   
        2018     30959.0   35953.0     10278.0    65941.0  245655.0  74080.0   
        2019     32002.0   35935.0     10335.0    65231.0  244664.0  74794.0   
        2020     34018.0   35959.0     10369.0    64606.0  243318.0  75379.0   
        2021     34465.0   36352.0     10554.0    64555.0  240800.0  75030.0   
        2022     35492.0   36409.0     10567.0    64010.0  239489.0  75844.0   
        2023     39152.0   36605.0     10712.0    63458.0  237602.0  77207.0   
        2024     42499.0   36626.0     10764.0    62929.0  236252.0  79149.0   
        2025     45961.0   36735.0     10918.0    62490.0  234647.0  81452.0   

                 ed_16_64  ed_0_15  altas_int  altas_ext  bajas_int  \
cod_mun periodo                                                       
02      2015     258810.0  62850.0     8816.0     1860.0    10322.0   
        2016     256947.0  61869.0     8222.0     2119.0     9746.0   
        2017     255674.0  60847.0     7825.0     2475.0     9379.0   
        2018     254450.0  60256.0     9724.0     3204.0    10809.0   
        2019     253910.0  59463.0     9241.0     4031.0    10143.0   
        2020     254323.0  58568.0     8395.0     2497.0     8473.0   
        2021     253828.0  57868.0     9301.0     4542.0     9844.0   
        2022     253112.0  57011.0    10422.0     6622.0    11199.0   
        2023     254063.0  56259.0    10480.0     6624.0    10979.0   
        2024     254496.0  55425.0    10211.0     7027.0    10772.0   
        2025     254732.0  54567.0        NaN        NaN        NaN   

                 bajas_ext  edu_primaria_inf  edu_superior  edu_secundaria_1  \
cod_mun periodo                                                                
02      2015        1452.0               NaN           NaN               NaN   
        2016        1524.0               NaN           NaN               NaN   
        2017        1704.0               NaN           NaN               NaN   
        2018        1915.0               NaN           NaN               NaN   
        2019        1640.0               NaN           NaN               NaN   
        2020        1236.0               NaN           NaN               NaN   
        2021        3740.0           68355.0       82318.0          119650.0   
        2022        2927.0           64743.0       84683.0          120992.0   
        2023        3414.0           62968.0       87128.0          121525.0   
        2024        3677.0           61196.0       89704.0          122289.0   
        2025           NaN               NaN           NaN               NaN   

                 edu_secundaria_2  act_inactivo  act_ocupado  act_parado  \
cod_mun periodo                                                            
02      2015                  NaN           NaN          NaN         NaN   
        2016                  NaN           NaN          NaN         NaN   
        2017                  NaN           NaN          NaN         NaN   
        2018                  NaN           NaN          NaN         NaN   
        2019                  NaN           NaN          NaN         NaN   
        2020                  NaN           NaN          NaN         NaN   
        2021              62437.0      139315.0     153971.0     35572.0   
        2022              62681.0      139847.0     160954.0     28155.0   
        2023              63794.0      143328.0     161566.0     26376.0   
        2024              64690.0      145183.0     163938.0     24524.0   
        2025                  NaN           NaN          NaN         NaN   

                 sp_cuen

In [116]:
df = df.set_index(["cod_mun", "periodo"])
df.update(vars_provincias)
df = df.reset_index()
# df = df.set_index(["municipio", "periodo"]).fillna(vars_provincias).reset_index()s
df.head(25)

,cod_mun,periodo,nombre,hombres,mujeres,total,n_ext,n_dif_ca,n_dif_prov,n_dif_mun,n_en_mun,ed_65+,ed_16_64,ed_0_15,altas_ext,altas_int,bajas_ext,bajas_int,edu_primaria_inf,edu_superior,edu_secundaria_1,edu_secundaria_2,act_inactivo,act_ocupado,act_parado,sp_cuenta_ajena_y_otros,sp_cuenta_propia,ae_agricultura_ganaderia_pesca,ae_construccion,ae_industria,ae_no_consta,ae_servicios,mnp_crecimiento_veg,mnp_fallecidos,mnp_matrimonios,mnp_nacimientos,cod_prov
0,02,2000,Albacete,180311.0,182952.0,363263.0,4229.0,33134.0,9017.0,72544.0,244339.0,64596.0,232715.0,65952.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
1,02,2001,Albacete,182826.0,184457.0,367283.0,7560.0,33336.0,9113.0,74643.0,242631.0,65957.0,236246.0,65080.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
2,02,2002,Albacete,185458.0,186329.0,371787.0,11968.0,33561.0,9160.0,74192.0,242906.0,67020.0,240043.0,64724.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
3,02,2003,Albacete,188139.0,188417.0,376556.0,15777.0,33995.0,9348.0,73829.0,243607.0,67861.0,243724.0,64971.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
4,02,2004,Albacete,189743.0,189705.0,379448.0,18509.0,34283.0,9451.0,73507.0,243698.0,68315.0,246642.0,64491.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
5,02,2005,Albacete,192624.0,192016.0,384640.0,22906.0,34816.0,9543.0,73198.0,244177.0,67913.0,252139.0,64588.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
6,02,2006,Albacete,194348.0,193310.0,387658.0,25073.0,35251.0,9675.0,72932.0,244727.0,69503.0,253474.0,64681.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
7,02,2007,Albacete,196772.0,195338.0,392110.0,28594.0,35833.0,9788.0,72551.0,245344.0,69580.0,257699.0,64831.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
8,02,2008,Albacete,199820.0,197673.0,397493.0,33652.0,36246.0,9867.0,72241.0,245487.0,69854.0,262632.0,65007.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02
9,02,2009,Albacete,201162.0,199729.0,400891.0,35488.0,36767.0,9972.0,71830.0,246834.0,70562.0,264992.0,65337.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02


# 6. Coordenadas

## 6.1. Municipios

In [117]:
coords = pd.read_csv("datos/coordenadas/MUNICIPIOS.csv", sep=";", decimal=",", encoding="latin1", dtype={"COD_PROV": str, "COD_INE": str})
coords = coords[coords["COD_PROV"].isin(PRE_CLM)]
coords["cod_mun"] = coords["COD_INE"].str[:5]
coords.head(10)

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD,cod_mun
51,02001000000,1020019,2010,02,Albacete,Abengibre,747,3076.9225,25257,2001000101,Abengibre,747,0743-4,-1.540796,39.209752,Detección automática,646.0,MDT,02001
52,02002000000,1020024,2020,02,Albacete,Alatoz,506,6385.3600,37234,2002000101,Alatoz,491,0767-1,-1.361010,39.094448,Detección automática,850.0,MDT,02002
53,02003000000,1020030,2001,02,Albacete,Albacete,175068,112698.9600,250011,2003000201,Albacete,167350,0790-2,-1.855678,38.996134,Detección automática,681.0,MDT,02003
54,02004000000,1020045,2030,02,Albacete,Albatana,653,3052.8200,22662,2004000101,Albatana,633,0843-4,-1.522800,38.571101,Detección automática,583.0,MDT,02004
55,02005000000,1020058,2040,02,Albacete,Alborea,688,7186.7600,36330,2005000101,Alborea,679,0744-1,-1.395985,39.279683,Detección automática,696.0,MDT,02005
56,02006000000,1020061,2050,02,Albacete,Alcadozo,638,9958.7000,53730,2006000101,Alcadozo,593,0842-2,-1.980883,38.648865,Detección automática,925.0,MDT,02006
57,02007000000,1020077,2060,02,Albacete,Alcalá del Júcar,1156,14682.4446,51494,2007000101,Alcalá del Júcar,663,0744-3,-1.429209,39.191681,Detección automática,539.0,MDT,02007
58,02008000000,1020083,2070,02,Albacete,Alcaraz,1281,37062.4100,290208,2008000101,Alcaraz,1085,0841-1,-2.491229,38.665085,Detección automática,958.0,MDT,02008
59,02009000000,1020096,2080,02,Albacete,Almansa,24603,53240.7900,93195,2009000101,Almansa,23816,0793-3,-1.093667,38.870070,Detección automática,685.0,MDT,02009
60,02010000000,1020100,2090,02,Albacete,Alpera,2320,17837.7900,56572,2010000101,Alpera,2302,0792-2,-1.230196,38.959504,Detección automática,850.0,MDT,02010


In [118]:
coords = coords[["COD_PROV", "cod_mun", "PROVINCIA", "NOMBRE_ACTUAL", "SUPERFICIE", "PERIMETRO", "CAPITAL", "LONGITUD_ETRS89_REGCAN95", "LATITUD_ETRS89_REGCAN95"]]
coords.head(10)

,COD_PROV,cod_mun,PROVINCIA,NOMBRE_ACTUAL,SUPERFICIE,PERIMETRO,CAPITAL,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95
51,02,02001,Albacete,Abengibre,3076.9225,25257,Abengibre,-1.540796,39.209752
52,02,02002,Albacete,Alatoz,6385.3600,37234,Alatoz,-1.361010,39.094448
53,02,02003,Albacete,Albacete,112698.9600,250011,Albacete,-1.855678,38.996134
54,02,02004,Albacete,Albatana,3052.8200,22662,Albatana,-1.522800,38.571101
55,02,02005,Albacete,Alborea,7186.7600,36330,Alborea,-1.395985,39.279683
56,02,02006,Albacete,Alcadozo,9958.7000,53730,Alcadozo,-1.980883,38.648865
57,02,02007,Albacete,Alcalá del Júcar,14682.4446,51494,Alcalá del Júcar,-1.429209,39.191681
58,02,02008,Albacete,Alcaraz,37062.4100,290208,Alcaraz,-2.491229,38.665085
59,02,02009,Albacete,Almansa,53240.7900,93195,Almansa,-1.093667,38.870070
60,02,02010,Albacete,Alpera,17837.7900,56572,Alpera,-1.230196,38.959504


In [119]:
print(coords[coords["cod_mun"] == "45199"]["NOMBRE_ACTUAL"].drop_duplicates())
print(df[df["cod_mun"] == "45199"]["nombre"].drop_duplicates())

6977    El Viso de San Juan
Name: NOMBRE_ACTUAL, dtype: str
23816    Viso de San Juan, El
Name: nombre, dtype: str


> CAMBIA EL FORMATO DE LOS NOMBRES CON ARTICULO

In [120]:
# df[df["municipio"].str.contains(",")]["municipio"].unique()

In [121]:
coords = coords[["cod_mun", "COD_PROV", "SUPERFICIE", "PERIMETRO", "LONGITUD_ETRS89_REGCAN95", "LATITUD_ETRS89_REGCAN95"]]
coords = coords.rename(columns={"COD_PROV": "cod_prov", "SUPERFICIE": "superficie", "PERIMETRO": "perimetro", "LONGITUD_ETRS89_REGCAN95": "lon", "LATITUD_ETRS89_REGCAN95": "lat"})
coords["superficie"] = coords["superficie"] / 100 # De hectáreas a kilometros cuadrados
coords.head()

,cod_mun,cod_prov,superficie,perimetro,lon,lat
51,02001,02,30.769225,25257,-1.540796,39.209752
52,02002,02,63.853600,37234,-1.361010,39.094448
53,02003,02,1126.989600,250011,-1.855678,38.996134
54,02004,02,30.528200,22662,-1.522800,38.571101
55,02005,02,71.867600,36330,-1.395985,39.279683


In [122]:
df = df.merge(coords[["cod_mun", "superficie", "perimetro", "lon", "lat"]], how="left", on="cod_mun")
df[df["cod_mun"] != "02"].head()

,cod_mun,periodo,nombre,hombres,mujeres,total,n_ext,n_dif_ca,n_dif_prov,n_dif_mun,n_en_mun,ed_65+,ed_16_64,ed_0_15,altas_ext,altas_int,bajas_ext,bajas_int,edu_primaria_inf,edu_superior,edu_secundaria_1,edu_secundaria_2,act_inactivo,act_ocupado,act_parado,sp_cuenta_ajena_y_otros,sp_cuenta_propia,ae_agricultura_ganaderia_pesca,ae_construccion,ae_industria,ae_no_consta,ae_servicios,mnp_crecimiento_veg,mnp_fallecidos,mnp_matrimonios,mnp_nacimientos,cod_prov,superficie,perimetro,lon,lat
26,02001,2000,Abengibre,479.0,497.0,976.0,7.0,47.0,14.0,141.0,767.0,219.0,633.0,124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
27,02001,2001,Abengibre,487.0,500.0,987.0,7.0,49.0,15.0,151.0,765.0,231.0,632.0,124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
28,02001,2002,Abengibre,478.0,494.0,972.0,13.0,49.0,15.0,155.0,740.0,230.0,621.0,121.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
29,02001,2003,Abengibre,480.0,479.0,959.0,17.0,49.0,15.0,149.0,729.0,229.0,615.0,115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
30,02001,2004,Abengibre,483.0,494.0,977.0,26.0,52.0,16.0,148.0,735.0,233.0,629.0,115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752


## 6.2. Provincias

Datos del [IGN](https://www.ign.es/resources/ane/Informacion_Geografica_Destacada/IGN_INFOGEO_PROVINCIAS.html)

In [123]:
df.loc[df["cod_mun"] == "02", "superficie"] = 14_931.74 # Albacete
df.loc[df["cod_mun"] == "13", "superficie"] = 19_812.04 # Ciudad Real
df.loc[df["cod_mun"] == "16", "superficie"] = 17_121.15 # Cuenca
df.loc[df["cod_mun"] == "19", "superficie"] = 12_213.17 # Guadalajara
df.loc[df["cod_mun"] == "45", "superficie"] = 15_369.34 # Toledo
df.loc[df["cod_mun"].isin(PRE_CLM), ["cod_mun", "nombre", "superficie"]].drop_duplicates()

,cod_mun,nombre,superficie
0,02,Albacete,14931.74
2288,13,Ciudad Real,19812.04
4966,16,Cuenca,17121.15
11180,19,Guadalajara,12213.17
18694,45,Toledo,15369.34


# 7. Guardar datos

In [124]:
df[df["cod_mun"] != "02"].head(26)

,cod_mun,periodo,nombre,hombres,mujeres,total,n_ext,n_dif_ca,n_dif_prov,n_dif_mun,n_en_mun,ed_65+,ed_16_64,ed_0_15,altas_ext,altas_int,bajas_ext,bajas_int,edu_primaria_inf,edu_superior,edu_secundaria_1,edu_secundaria_2,act_inactivo,act_ocupado,act_parado,sp_cuenta_ajena_y_otros,sp_cuenta_propia,ae_agricultura_ganaderia_pesca,ae_construccion,ae_industria,ae_no_consta,ae_servicios,mnp_crecimiento_veg,mnp_fallecidos,mnp_matrimonios,mnp_nacimientos,cod_prov,superficie,perimetro,lon,lat
26,02001,2000,Abengibre,479.0,497.0,976.0,7.0,47.0,14.0,141.0,767.0,219.0,633.0,124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
27,02001,2001,Abengibre,487.0,500.0,987.0,7.0,49.0,15.0,151.0,765.0,231.0,632.0,124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
28,02001,2002,Abengibre,478.0,494.0,972.0,13.0,49.0,15.0,155.0,740.0,230.0,621.0,121.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
29,02001,2003,Abengibre,480.0,479.0,959.0,17.0,49.0,15.0,149.0,729.0,229.0,615.0,115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
30,02001,2004,Abengibre,483.0,494.0,977.0,26.0,52.0,16.0,148.0,735.0,233.0,629.0,115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
31,02001,2005,Abengibre,483.0,494.0,977.0,47.0,49.0,16.0,146.0,719.0,229.0,635.0,113.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
32,02001,2006,Abengibre,476.0,484.0,960.0,66.0,45.0,15.0,150.0,684.0,235.0,610.0,115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
33,02001,2007,Abengibre,490.0,475.0,965.0,74.0,44.0,15.0,155.0,677.0,240.0,621.0,104.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
34,02001,2008,Abengibre,478.0,462.0,940.0,82.0,40.0,14.0,143.0,661.0,230.0,619.0,91.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752
35,02001,2009,Abengibre,473.0,457.0,930.0,92.0,39.0,14.0,140.0,645.0,236.0,601.0,93.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,02,30.769225,25257.0,-1.540796,39.209752


In [125]:
from pathlib import Path

path = Path("data/pob_clm.csv")
path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(path, index=False)

In [126]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24024 entries, 0 to 24023
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   cod_mun                         24024 non-null  str    
 1   periodo                         24024 non-null  int64  
 2   nombre                          24024 non-null  str    
 3   hombres                         24023 non-null  float64
 4   mujeres                         24023 non-null  float64
 5   total                           24023 non-null  float64
 6   n_ext                           24002 non-null  float64
 7   n_dif_ca                        24002 non-null  float64
 8   n_dif_prov                      24002 non-null  float64
 9   n_dif_mun                       24002 non-null  float64
 10  n_en_mun                        24002 non-null  float64
 11  ed_65+                          24002 non-null  float64
 12  ed_16_64                        24002 non-n